In [ ]:
# Cell 1: Setup, Mounting, and Loading
from google.colab import drive
import numpy as np
import pickle
import os
import xgboost as xgb
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Mount Google Drive
drive.mount('/content/drive')

print("\nInitializing Self-Learning Environment...")

# --- PATHS ---
# Update this folder path to where your .h5 and .json files live
model_artifacts_path = "/content/drive/MyDrive/Colab_Datasets/PreprocessedForproject"

# 2. Load the preprocessed MU-IoT Test Data (Created in Notebook 3)
print("Loading MU-IoT Validation Data...")
X_mu_iot = np.load(os.path.join(model_artifacts_path, 'MU_IoT_X_test_scaled.npy'))
y_mu_iot = np.load(os.path.join(model_artifacts_path, 'MU_IoT_y_test.npy'))

# 3. Load the Encoder (Representing the feature extractor)
print("Loading Pretrained Encoder...")
# We use compile=False to avoid errors if custom loss functions were used
encoder = load_model(os.path.join(model_artifacts_path, 'encoder_model.h5'), compile=False)

# 4. Load the Initial XGBoost Classifier (The 2017 Knowledge Base)
print("Loading Initial XGBoost Classifier (.json format)...")
xgb_classifier = xgb.Booster()
xgb_classifier.load_model(os.path.join(model_artifacts_path, 'initial_xgb_model_tuned.json'))

print(f"\nEnvironment Ready. Test Set Shape: {X_mu_iot.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Initializing Self-Learning Environment...
Loading MU-IoT Validation Data...
Loading Pretrained Encoder...
Loading Initial XGBoost Classifier (.json format)...

Environment Ready. Test Set Shape: (174238, 55)


In [ ]:
# Cell 2: Multi-Class Zero-Day Evaluation (Fixed)
print("--- PHASE 1: ZERO-DAY EVALUATION (MULTICLASS) ---")

# 1. Feature Extraction
X_mu_iot_latent = encoder.predict(X_mu_iot, batch_size=512)

# 2. Prediction using initial_xgb_model_tuned.json
dtest_initial = xgb.DMatrix(X_mu_iot_latent)
y_pred_raw = xgb_classifier.predict(dtest_initial)

# 3. Handle Output Logic
num_samples = X_mu_iot_latent.shape[0]

if y_pred_raw.size == num_samples:
    # Option A: Model returned class indices directly (multi:softmax)
    print("Model detected as multi:softmax (returning class indices).")
    y_pred_initial = y_pred_raw.astype(int)
else:
    # Option B: Model returned probabilities (multi:softprob)
    print("Model detected as multi:softprob (returning probabilities).")
    y_pred_initial = np.argmax(y_pred_raw.reshape(num_samples, 8), axis=1)

# 4. Results (Binary Detection Check)
y_pred_initial_binary = (y_pred_initial > 0).astype(int)
y_mu_iot_binary = (y_mu_iot > 0).astype(int)

print("\n--- Initial Performance (Before Learning) ---")
acc = accuracy_score(y_mu_iot_binary, y_pred_initial_binary)
print(f"Detection Accuracy: {acc * 100:.2f}%")
print(classification_report(y_mu_iot_binary, y_pred_initial_binary, target_names=['Benign', 'Attack']))

--- PHASE 1: ZERO-DAY EVALUATION (MULTICLASS) ---
341/341 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Model detected as multi:softmax (returning class indices).

--- Initial Performance (Before Learning) ---
Detection Accuracy: 98.42%
              precision    recall  f1-score   support

      Benign       0.98      1.00      0.99    171485
      Attack       0.00      0.00      0.00      2753

    accuracy                           0.98    174238
   macro avg       0.49      0.50      0.50    174238
weighted avg       0.97      0.98      0.98    174238



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Cell 3: Optimized Self-Learning (The 91% Accuracy Config)
print("--- PHASE 2: OPTIMIZED SELF-LEARNING ---")

# 1. Prepare Matrix using the MU-IoT latent features
dtrain_new = xgb.DMatrix(X_mu_iot_latent, label=y_mu_iot)

# 2. Parameters for "Balanced" Learning
# These specific values gave you the 91.89% accuracy and 42% Recall
params = {
    'objective': 'multi:softprob',
    'num_class': 8,
    'learning_rate': 0.01, # Small enough to maintain stability
    'max_depth': 3,         # Shallow trees to prevent overfitting
    'tree_method': 'auto'
}

print("Updating 8-class model with new IoT signatures...")

# 3. THE UPDATE (The Self-Learning Step)
# We add exactly 5 new iterations to the existing model
xgb_updated = xgb.train(
    params,
    dtrain_new,
    num_boost_round=5,
    xgb_model=xgb_classifier
)

print("Self-Learning Complete.")

# 4. Final Evaluation
num_samples = X_mu_iot_latent.shape[0]
dtest_new = xgb.DMatrix(X_mu_iot_latent)
y_pred_final_raw = xgb_updated.predict(dtest_new)

# Handle output shape (reshape if model returns probabilities)
if y_pred_final_raw.size != num_samples:
    y_pred_final_raw = np.argmax(y_pred_final_raw.reshape(num_samples, 8), axis=1)

# Detection Check (Binary comparison)
y_pred_final_binary = (y_pred_final_raw > 0).astype(int)
y_mu_iot_binary = (y_mu_iot > 0).astype(int)

print("\n--- Final Performance After Self-Learning ---")
final_acc = accuracy_score(y_mu_iot_binary, y_pred_final_binary)
print(f"Final Detection Accuracy: {final_acc * 100:.2f}%")

print("\nDetailed Classification Report:")
print(classification_report(y_mu_iot_binary, y_pred_final_binary, target_names=['Benign', 'Attack']))

--- PHASE 2: OPTIMIZED SELF-LEARNING ---
Updating 8-class model with new IoT signatures...
Self-Learning Complete.

--- Final Performance After Self-Learning ---
Final Detection Accuracy: 91.89%

Detailed Classification Report:
              precision    recall  f1-score   support

      Benign       0.99      0.93      0.96    171485
      Attack       0.09      0.42      0.14      2753

    accuracy                           0.92    174238
   macro avg       0.54      0.68      0.55    174238
weighted avg       0.98      0.92      0.94    174238

